# Introduction to privJedAI

This notebook will guide you throughout all the possible methods and how to use them by our open-source library privJedAI.

In [1]:
%pip install privjedai


Found existing installation: privJedAI 0.0.2
Can't uninstall 'privJedAI'. No files were found to uninstall.
Note: you may need to restart the kernel to use updated packages.
Obtaining file:///home/lefteris/privJedAI
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for privJedAI (pyproject.toml) ... done
  Created wheel for privJedAI: filename=privjedai-0.0.2-0.editable-py3-none-any.whl size=8812 sha256=30b6a9fd8d9fdf2fb0765447da93a92724310d2f19c0d9b09f1b56ea8c099721
  Stored in directory: /tmp/pip-ephem-wheel-cache-31o_8nio/wheels/3d/c5/59/5bd4e07c02455ba64adf12d849d7aadca35951ed88bc75a965
Successfully built privJedAI
  Attempting uninstall: privJedAI
    Found existing installation: privJedAI 0.0.2
    Can't uninstall 'privJedAI'. No files were found to uninstall.
Note: you may need to restart the kernel to u

## Import Dataset Abt Buy Clean

Below we load the two different datasets and their ground truth. privJedAI needs the indices of the pairs for each dataset and not their id. Here we present also how to preprocess a ground truth that contains id1-id2 pairs to index1-index2 pairs.

In [1]:
import pandas as pd

def load_dataset(path, file, sep):
    return pd.read_csv(f"../{path}/{file}.csv", sep=sep)

def load_ground_truth(path, sep, d1, d2): 
    gt = pd.read_csv(f'../{path}/gtclean.csv' , sep=sep)
    df_a = d1.reset_index().rename(columns={"index": "index_A"})
    df_b = d2.reset_index().rename(columns={"index": "index_B"})

    df_a = df_a[["index_A", "id"]]
    df_b = df_b[["index_B", "id"]]

    gt_index = gt.merge(left_on='D1', right=df_a, right_on='id')

    gt_index = gt_index.drop(columns=['id', 'D1'])
    gt_index.columns = ['D2', 'D1']

    gt_index = gt_index.merge(left_on='D2', right=df_b, right_on='id')
    gt_index = gt_index.drop(columns=['id', 'D2'])
    gt_index.columns = ['D1', 'D2']
    d1 = d1.astype(str)
    d2 = d2.astype(str)

    return gt_index, d1, d2


In [3]:

DIR = "D2"
PATH = f"data/ccer/{DIR}"
FILE = 'abtclean'
FILE2 = 'buyclean'
attributes = ['name']
SEP = "|"

abt = load_dataset(PATH, FILE, SEP)
buy = load_dataset(PATH, FILE2, SEP)

gt_index, abt, buy = load_ground_truth(PATH, SEP, abt, buy)

## Encode data and build bloom filters

Each party agree in an exact configuration and then encode locally their data.
Those encoded data are then shared to a third party to proceed with record linkage.

In [4]:
from privjedai.encoder import BloomFilterConfig, BloomEncodedData, BloomFilter

bloom_filter_configuration = {
    "size" : 1024,
    "offset" : 0,
    "num_hashes" : 15,
    "hashing_type": "salted_qgrams",
    "salt" : "",
    "attributes": ['name'],
    "qgrams": 4
}

# Abt Owner Encodes Dataset

In [5]:
config = BloomFilterConfig(**bloom_filter_configuration)
bloom_generator = BloomFilter(config)

## The two parties encode their datasets and save them to disk.
## The encoded datasets are then shared with the other party and used for the matching process.
encoded_d1 = bloom_generator.encode(abt)
encoded_d1.to_file(f"dataset_1.pkl")

Encoding Data with attributes ['name']:   0%|          | 0/1076 [00:00<?, ?it/s]

## Buy Owner Encodes Dataset

In [6]:
bloom_generator_buy = BloomFilter(config)
encoded_d2 = bloom_generator_buy.encode(buy)
encoded_d2.to_file(f"dataset_2.pkl")

Encoding Data with attributes ['name']:   0%|          | 0/1064 [00:00<?, ?it/s]

## Trusted Third Party: Linking Phase

In [7]:
# Third party loads the encoded datasets and performs the matching process.
encoded_data = BloomEncodedData.from_file("dataset_1.pkl", "dataset_2.pkl")


# Ground truth must be explicitly set for the evaluation process. 
# This is done by providing the indices of the matching records in the original datasets.r
encoded_data.set_ground_truth(gt_index)

## Blocking with privJedAI

In privJedAI we have 2 different implementations of Hamming LSH and a FAISS implementation.

### BitBlocker

In [8]:
from privjedai.blocking import BitBlocker

blocker = BitBlocker(psi = 8,
            lambda_ = 24,
            seed = 42)

blocks = blocker.build_blocks(encoded_data=encoded_data)


_ = blocker.evaluate(blocks)


***************************************************************************************************************************
                                         Method:  BitBlocker
***************************************************************************************************************************
Method name: BitBlocker
Parameters: 
Runtime: 0.2469 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.53% 
	Recall:        88.35%
	F1-score:       1.06%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### LSHBlocker


In [10]:
from privjedai.blocking import LSHBlocker

blocker = LSHBlocker(psi = 8,
            lambda_ = 24,
            seed = 42,
            prune_ratio = 0.8)

blocks = blocker.build_blocks(encoded_data=encoded_data)
_ = blocker.evaluate(blocks)

***************************************************************************************************************************
                                         Method:  LSHBlocker
***************************************************************************************************************************
Method name: LSHBlocker
Parameters: 
	psi: 8
	lambda_: 24
	prune_ratio: 0.8
	prune_sample: 1000
	seed: 42
Runtime: 1.0429 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.09% 
	Recall:        91.07%
	F1-score:       0.18%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### FAISSBlocking

In [9]:
from privjedai.blocking import FAISSBlocking

blocker = FAISSBlocking(index_type='flat')

blocks = blocker.build_blocks(encoded_data=encoded_data, top_k=20)
_ = blocker.evaluate(blocks)

***************************************************************************************************************************
                                         Method:  FAISS Blocking
***************************************************************************************************************************
Method name: FAISS Blocking
Parameters: 
Runtime: 0.0288 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:     72.93% 
	Recall:        72.93%
	F1-score:      72.93%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Meta-blocking Techniques

Those above are all standard blocking techniques. privJedAI also implements meta-blocking methods. It leverages comparison cleaning methods from ER and implements them for bitarrays. A block is a set of adjacent active bits of a bitarray.

In [10]:
from privjedai.comparison_cleaning import ( 
    WeightedEdgePruning,
    WeightedNodePruning,
    CardinalityEdgePruning,
    CardinalityNodePruning
)

cc = CardinalityEdgePruning(weighting_scheme='CN-CBS')

cc_blocks = cc.process(encoded_data, adjacent_bits=2)

_ = cc.evaluate(cc_blocks)

***************************************************************************************************************************
                                         Method:  Cardinality Edge Pruning
***************************************************************************************************************************
Method name: Cardinality Edge Pruning
Parameters: 
	Node centric: False
	Weighting scheme: CN-CBS
Runtime: 2.6905 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:      0.31% 
	Recall:        98.87%
	F1-score:       0.61%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Matching

After filtering our datasets we can use a similarity function and match the possible candidate pairs.

In [11]:
from privjedai.matching import Matcher
import numpy as np
matcher = Matcher(batch_size = 10_000,
                  threshold = 0.6,
                  metric='dice')

matches = matcher.predict(encoded_data=encoded_data, blocks=blocks)

_ = matcher.evaluate(matches)

Predicting batches:   0%|          | 0/1 [00:00<?, ?it/s]

***************************************************************************************************************************
                                         Method:  Matcher
***************************************************************************************************************************
Method name: Matcher
Parameters: 
	batch_size: 10000
	threshold: 0.6
	metric: dice
	attributes: None
Runtime: 0.0927 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:     76.45% 
	Recall:        69.27%
	F1-score:      72.68%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Clustering

To eliminate possible conflicts on the matching pairs, we provide multiple clustering techniques.

In [12]:
from privjedai.clustering import KiralyMSMApproximateClustering

clusterer = KiralyMSMApproximateClustering()

clusters = clusterer.process(matches, encoded_data=encoded_data)

_ = clusterer.evaluate(clusters)

***************************************************************************************************************************
                                         Method:  Kiraly MSM Approximate Clustering
***************************************************************************************************************************
Method name: Kiraly MSM Approximate Clustering
Parameters: 
	Similarity Threshold: 0.1
Runtime: 0.0271 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:     91.92% 
	Recall:        67.39%
	F1-score:      77.77%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Additonal Features

Concurrent matching using ray

In [13]:
from privjedai.ray.matching import Matcher
import numpy as np
matcher = Matcher(batch_size = 10_000,
                  threshold = 0.6,
                  metric='dice',
                  workers=10)

matches = matcher.predict(encoded_data=encoded_data, blocks=blocks)

_ = matcher.evaluate(matches)

/home/conda/miniconda3/envs/pprl/lib/python3.10/site-packages/ray/_private/worker.py:2062: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Predicting batches:   0%|          | 0/1 [00:00<?, ?it/s]

***************************************************************************************************************************
                                         Method:  Matcher
***************************************************************************************************************************
Method name: Matcher
Parameters: 
	batch_size: 10000
	threshold: 0.6
	metric: dice
	attributes: None
Runtime: 0.7185 seconds
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Performance:
	Precision:     76.45% 
	Recall:        69.27%
	F1-score:      72.68%
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## GPU-Accelaration

In [ ]:
from privjedai.gpu.matching import Matcher
from privjedai.gpu.clustering import KiralyMSMApproximateClustering

matcher = Matcher(batch_size = 10_000,
                  threshold = 0.6,
                  metric='dice',
                  )

matches = matcher.predict(encoded_data=encoded_data, blocks=blocks)

_ = matcher.evaluate(matches, verbose=True)

clusterer = KiralyMSMApproximateClustering()
clusters = clusterer.process(matches, encoded_data=encoded_data)
_ = clusterer.evaluate(clusters)
